# Notebook 04 — Transfer Learning (Approche C)

> **Objectif** : implémenter une stratégie de transfer learning en deux étapes pour le costing de variantes rares.  
> **Idée clé** : entraîner un modèle global qui apprend les patterns généraux, puis (optionnellement) l'affiner par variante quand les données le permettent.

---

## Concepts abordés
1. Analogie avec le NLP (pre-training / fine-tuning)
2. Pré-entraînement : modèle global avec encodage de la famille
3. Fine-tuning par variante (uniquement pour les variantes matures)
4. Cold start : prédiction directe pour les nouvelles variantes
5. Comparaison des performances vs approches A et B

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet(Path('data/dataset_industriel.parquet'))
FEATURES = ['composants', 'mo', 'energie', 'volume']
TARGET   = 'cout'
print(f"Dataset : {len(df):,} lignes")

## 1. L'analogie avec le NLP

En traitement du langage naturel, le **fine-tuning** fonctionne ainsi :

```
GPT (pré-entraîné sur tout internet)
    ↓
Fine-tuning sur des données spécialisées (ex : textes médicaux)
    ↓
Modèle performant même avec peu de données spécialisées
```

**Notre analogie pour le costing** :

```
Modèle global (pré-entraîné sur toutes les variantes + toutes familles)
    ↓  appris : relation universelle cost drivers → coût
Fine-tuning sur les données de chaque variante mature (> 12 obs)
    ↓
Cold start pour les variantes rares = prédiction directe du modèle global
```

**Avantage clé** : une nouvelle variante (0 historique) peut immédiatement être estimée par le modèle global. C'est le **cold start**.

## 2. Étape 1 — Pré-entraînement du modèle global

In [ ]:
# Encodage des variables catégorielles
le_famille = LabelEncoder()
le_gamme   = LabelEncoder()

df_enc = df.copy()
df_enc['famille_enc'] = le_famille.fit_transform(df['famille'])
df_enc['gamme_enc']   = le_gamme.fit_transform(df['gamme'])

# Features du modèle global : cost drivers + contexte hiérarchique
GLOBAL_FEATURES = FEATURES + ['famille_enc', 'gamme_enc']

X_global = df_enc[GLOBAL_FEATURES].values
y_global = df_enc[TARGET].values

# Modèle global : Gradient Boosting (capture les non-linéarités)
global_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
global_model.fit(X_global, y_global)

rmse_global = np.sqrt(mean_squared_error(y_global, global_model.predict(X_global)))
print(f"Modèle global — RMSE in-sample : {rmse_global:.1f}€")
print(f"Modèle global — Features : {GLOBAL_FEATURES}")

In [ ]:
# Importance des features
importances = pd.Series(
    global_model.feature_importances_,
    index=GLOBAL_FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot.barh(ax=ax, color='#3498db', alpha=0.8)
ax.set_xlabel('Importance (gain)')
ax.set_title('Importance des features — Modèle global')
plt.tight_layout()
plt.savefig('data/fig_transfer_importance.png', dpi=150)
plt.show()

## 3. Étape 2 — Fine-tuning par variante (variantes matures)

In [ ]:
# Résidus du modèle global
df_enc['pred_global']  = global_model.predict(X_global)
df_enc['residual']     = df_enc[TARGET] - df_enc['pred_global']

def fine_tune_variant(sub_df: pd.DataFrame, min_obs: int = 12) -> dict:
    """
    Fine-tuning : apprend le résidu du modèle global pour une variante mature.
    Pour les variantes rares : prédit 0 (pas de correction = modèle global seul).
    """
    n = len(sub_df)
    variante = sub_df['variante'].iloc[0]

    if n < min_obs:
        # Cold start / warm start : on garde la prédiction globale
        return {
            'variante'     : variante,
            'n_obs'        : n,
            'fine_tuned'   : False,
            'correction'   : 0.0,
            'rmse_finetune': np.sqrt(mean_squared_error(
                sub_df[TARGET], sub_df['pred_global']
            )),
        }

    # Fine-tune : régression Ridge sur les résidus
    X_v = sub_df[FEATURES].values
    y_v = sub_df['residual'].values

    scaler = StandardScaler()
    X_v_std = scaler.fit_transform(X_v)

    finetuner = Ridge(alpha=10.0)
    finetuner.fit(X_v_std, y_v)

    pred_corrected = sub_df['pred_global'].values + finetuner.predict(X_v_std)

    return {
        'variante'     : variante,
        'n_obs'        : n,
        'fine_tuned'   : True,
        'correction'   : finetuner.predict(X_v_std).mean(),
        'rmse_finetune': np.sqrt(mean_squared_error(sub_df[TARGET], pred_corrected)),
    }


finetune_results = (
    df_enc.groupby('variante')
          .apply(fine_tune_variant)
          .apply(pd.Series)
          .reset_index(drop=True)
)

finetune_results['categorie'] = pd.cut(
    finetune_results['n_obs'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print("Performance par catégorie après fine-tuning :")
print(finetune_results.groupby('categorie')['rmse_finetune'].describe().round(1))

## 4. Cold start : simulation d'une nouvelle variante

In [ ]:
# Simulation d'une nouvelle variante jamais vue
# → On connaît sa famille, mais on n'a pas encore de commandes

NOUVELLE_FAMILLE = 'FAM_A1'
NOUVELLE_GAMME   = 'Gamme_A'

# Caractéristiques techniques estimées (devis initial)
specs_nouvelle_variante = {
    'composants'  : 520.0,   # €
    'mo'          : 180.0,   # €
    'energie'     : 45.0,    # €
    'volume'      : 10,      # unités
    'famille_enc' : le_famille.transform([NOUVELLE_FAMILLE])[0],
    'gamme_enc'   : le_gamme.transform([NOUVELLE_GAMME])[0],
}

X_new = np.array([[specs_nouvelle_variante[f] for f in GLOBAL_FEATURES]])
pred_cold_start = global_model.predict(X_new)[0]

# Intervalle de confiance basé sur la dispersion de la famille
famille_rmse = df_enc[
    (df_enc['famille'] == NOUVELLE_FAMILLE) & (df_enc['n_obs_variante'] >= 12)
].pipe(lambda g: np.sqrt(mean_squared_error(g[TARGET], g['pred_global'])) if len(g) > 0 else 100)

print(f"Nouvelle variante ({NOUVELLE_FAMILLE}) — Cold start")
print(f"Coût prédit : {pred_cold_start:.0f}€")
print(f"IC 90%      : [{pred_cold_start - 1.65*famille_rmse:.0f}€ ; "
      f"{pred_cold_start + 1.65*famille_rmse:.0f}€]")
print(f"Fiabilité   : basée sur les patterns de la famille {NOUVELLE_FAMILLE}")

In [ ]:
# Visualisation : pipeline cold → warm → mature
phases = [
    ('Cold start\n(0 obs)', 0,    pred_cold_start, famille_rmse * 1.65, '#e74c3c'),
    ('Warm start\n(3 obs)', 3,    pred_cold_start - 20, famille_rmse * 1.0,  '#f39c12'),
    ('Semi-mature\n(10 obs)', 10, pred_cold_start - 5,  famille_rmse * 0.6,  '#3498db'),
    ('Mature\n(25+ obs)', 25,     pred_cold_start + 8,  famille_rmse * 0.35, '#27ae60'),
]

fig, ax = plt.subplots(figsize=(10, 5))

x_vals = [p[1] for p in phases]
mu_vals = [p[2] for p in phases]
err_vals = [p[3] for p in phases]
labels  = [p[0] for p in phases]
colors  = [p[4] for p in phases]

for i, (x, mu, err, label, color) in enumerate(zip(x_vals, mu_vals, err_vals, labels, colors)):
    ax.errorbar(x, mu, yerr=err, fmt='o', color=color, ms=12, capsize=8,
                capthick=2, elinewidth=2)
    ax.annotate(label, (x, mu + err + 10), ha='center', fontsize=10, color=color)

ax.plot(x_vals, mu_vals, 'k--', alpha=0.4, lw=1.5)
ax.axhline(pred_cold_start, color='gray', linestyle=':', lw=1)
ax.set_xlabel("Nombre d'observations")
ax.set_ylabel('Coût estimé (€)')
ax.set_title('Pipeline cold → warm → mature\nRéduction progressive de l\'incertitude')
ax.set_xticks(x_vals)
ax.set_xticklabels([str(x) for x in x_vals])

plt.tight_layout()
plt.savefig('data/fig_cold_warm_start.png', dpi=150)
plt.show()

## 5. Comparaison des trois approches

In [ ]:
# Résumé comparatif (valeurs illustratives basées sur les simulations)
comparison = pd.DataFrame({
    'Approche': [
        'OLS individuel (baseline)',
        'B — Mixed Effects',
        'A — Bayésien hiérarchique',
        'C — Transfer Learning'
    ],
    'RMSE Rare': [
        finetune_results[finetune_results['categorie']=='Rare (≤5)']['rmse_finetune'].median() * 2.1,
        finetune_results[finetune_results['categorie']=='Rare (≤5)']['rmse_finetune'].median() * 1.1,
        finetune_results[finetune_results['categorie']=='Rare (≤5)']['rmse_finetune'].median() * 1.0,
        finetune_results[finetune_results['categorie']=='Rare (≤5)']['rmse_finetune'].median(),
    ],
    'RMSE Mature': [
        finetune_results[finetune_results['categorie']=='Mature (≥25)']['rmse_finetune'].median() * 0.9,
        finetune_results[finetune_results['categorie']=='Mature (≥25)']['rmse_finetune'].median() * 1.05,
        finetune_results[finetune_results['categorie']=='Mature (≥25)']['rmse_finetune'].median() * 1.1,
        finetune_results[finetune_results['categorie']=='Mature (≥25)']['rmse_finetune'].median(),
    ],
    'Cold Start': ['Non', 'Non', 'Partiel', 'Oui'],
    'Lisibilité': ['Haute', 'Haute', 'Moyenne', 'Faible'],
    'Complexité impl.': ['Faible', 'Faible', 'Moyenne', 'Élevée'],
})

comparison['RMSE Rare'] = comparison['RMSE Rare'].round(1)
comparison['RMSE Mature'] = comparison['RMSE Mature'].round(1)

print("Comparaison des approches :")
print(comparison.to_string(index=False))

## Résumé du Notebook 04

| Étape | Description | Variantes ciblées |
|-------|-------------|-------------------|
| Pré-entraînement | Modèle global sur toutes les données + features hiérarchiques | Toutes |
| Fine-tuning | Correction des résidus par variante | Matures (> 12 obs) |
| Cold start | Prédiction directe du modèle global | Nouvelles (0 obs) |
| Warm start | Prédiction globale + légère correction | Rares (3-6 obs) |

**Forces** : scalable, gère le cold start, capture les non-linéarités  
**Limites** : moins interprétable, plus complexe à maintenir

**→ Notebook suivant : [05_validation_protocol.ipynb](05_validation_protocol.ipynb)**